# 02 - Baseline Gate Cho Agent 1

Notebook này chạy baseline sau `01_data_preparation_eda.ipynb`.

Core baselines:
- `B1_customer_median_backoff`
- `B1R_recency_last5_median_backoff`

Due-date clamp không còn là baseline chính. Nếu cần, nó chỉ nên giữ như sanity check phụ.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.options.display.max_columns = 100
pd.options.display.float_format = "{:,.4f}".format

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "reference_notebooks").is_dir() and (parent / ".gitignore").is_file()
)
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

SNAPSHOT_FEATURES_PATH = PROCESSED_DIR / "rolling_snapshot_features.csv"
CURRENT_OPEN_FEATURES_PATH = PROCESSED_DIR / "current_open_invoice_features.csv"

BASELINE_PREDICTIONS_PATH = PROCESSED_DIR / "baseline_snapshot_predictions.csv"
BASELINE_INVOICE_METRICS_PATH = PROCESSED_DIR / "baseline_invoice_metrics.csv"
BASELINE_SEGMENT_METRICS_PATH = PROCESSED_DIR / "baseline_segment_metrics.csv"
BASELINE_CASH_BY_CUTOFF_PATH = PROCESSED_DIR / "baseline_cash_by_cutoff.csv"
BASELINE_CASH_METRICS_PATH = PROCESSED_DIR / "baseline_cash_metrics.csv"
CURRENT_OPEN_BASELINE_PATH = PROCESSED_DIR / "current_open_baseline_forecast.csv"

HORIZONS = [30, 60, 90]
DATE_COLUMNS = ["cutoff_date", "issue_date", "due_date", "paid_date", "target_paid_date"]
BASELINES = {
    "B1_customer_median_backoff": "pred_b1_remaining_days",
    "B1R_recency_last5_median_backoff": "pred_b1r_remaining_days",
}

## 1. Load dữ liệu đã xử lý

Notebook này chỉ đọc các artifact từ `01_data_preparation_eda.ipynb`.

In [ ]:
def read_processed_csv(path):
    frame = pd.read_csv(path, low_memory=False)
    for col in DATE_COLUMNS:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")
    return frame


snapshot_features = read_processed_csv(SNAPSHOT_FEATURES_PATH)
current_open_features = read_processed_csv(CURRENT_OPEN_FEATURES_PATH)

required_snapshot_columns = [
    "cutoff_date",
    "time_split",
    "invoice_id",
    "customer_id",
    "amount",
    "days_until_due_at_T",
    "remaining_days_at_T",
    "customer_invoice_count_hist",
    "customer_median_delay_hist",
    "customer_recency_delay_median_hist",
    "term_median_delay_hist",
    "global_median_delay_hist",
]
missing_columns = [col for col in required_snapshot_columns if col not in snapshot_features.columns]
if missing_columns:
    raise ValueError(f"Thiếu cột từ 01_data_preparation_eda.ipynb: {missing_columns}")

print(f"Snapshot rows: {len(snapshot_features):,}")
print(f"Cutoffs: {snapshot_features['cutoff_date'].nunique():,}")
print(f"Current open invoices: {len(current_open_features):,}")
snapshot_features.head()

## 2. Tạo baseline prediction

- `B1`: customer median delay với hierarchical backoff.
- `B1R`: recency last-5 median delay với backoff.
- Cả hai đều dự báo `remaining_days_at_T`.

In [ ]:
def positive_ceil_days(series):
    values = pd.to_numeric(series, errors="coerce").fillna(0).clip(lower=0)
    return np.ceil(values).astype("int64")


def add_predicted_paid_dates(frame, pred_col, output_col):
    frame[output_col] = pd.to_datetime(frame["cutoff_date"]) + pd.to_timedelta(
        positive_ceil_days(frame[pred_col]),
        unit="D",
    )
    return frame


def attach_baseline_predictions(frame):
    frame = frame.copy()
    days_until_due = pd.to_numeric(frame["days_until_due_at_T"], errors="coerce").fillna(0)

    customer_delay = pd.to_numeric(frame["customer_median_delay_hist"], errors="coerce")
    recency_delay = pd.to_numeric(frame["customer_recency_delay_median_hist"], errors="coerce")
    term_delay = pd.to_numeric(frame["term_median_delay_hist"], errors="coerce")
    global_delay = pd.to_numeric(frame["global_median_delay_hist"], errors="coerce")

    pred_b1_delay = customer_delay.fillna(term_delay).fillna(global_delay).fillna(0)
    pred_b1r_delay = recency_delay.fillna(customer_delay).fillna(term_delay).fillna(global_delay).fillna(0)

    frame["pred_b1_delay_days"] = pred_b1_delay
    frame["pred_b1_remaining_days"] = (days_until_due + pred_b1_delay).clip(lower=0)
    frame["pred_b1_source"] = np.where(
        pd.to_numeric(frame["customer_invoice_count_hist"], errors="coerce").fillna(0).gt(0),
        "customer_median_delay",
        "term_global_backoff",
    )

    frame["pred_b1r_delay_days"] = pred_b1r_delay
    frame["pred_b1r_remaining_days"] = (days_until_due + pred_b1r_delay).clip(lower=0)
    frame["pred_b1r_source"] = np.where(
        pd.to_numeric(frame["customer_invoice_count_hist"], errors="coerce").fillna(0).gt(0),
        "recency_last5_median",
        "term_global_backoff",
    )

    frame = add_predicted_paid_dates(frame, "pred_b1_remaining_days", "pred_b1_paid_date")
    frame = add_predicted_paid_dates(frame, "pred_b1r_remaining_days", "pred_b1r_paid_date")
    return frame


baseline_predictions = attach_baseline_predictions(snapshot_features)
current_open_baseline = attach_baseline_predictions(current_open_features)

prediction_priority = [
    "cutoff_date",
    "time_split",
    "invoice_id",
    "customer_id",
    "amount",
    "due_date",
    "target_paid_date",
    "remaining_days_at_T",
    "days_until_due_at_T",
    "is_overdue_at_T",
    "customer_history_sufficient",
    "pred_b1_remaining_days",
    "pred_b1_paid_date",
    "pred_b1_delay_days",
    "pred_b1_source",
    "pred_b1r_remaining_days",
    "pred_b1r_paid_date",
    "pred_b1r_delay_days",
    "pred_b1r_source",
]
baseline_predictions.loc[:, [c for c in prediction_priority if c in baseline_predictions.columns]].to_csv(
    BASELINE_PREDICTIONS_PATH, index=False
)

current_open_priority = [
    "cutoff_date",
    "invoice_id",
    "customer_id",
    "name_customer",
    "amount",
    "due_date",
    "days_until_due_at_T",
    "days_past_due_at_T",
    "is_overdue_at_T",
    "due_before_issue",
    "pred_b1_remaining_days",
    "pred_b1_paid_date",
    "pred_b1_delay_days",
    "pred_b1_source",
    "pred_b1r_remaining_days",
    "pred_b1r_paid_date",
    "pred_b1r_delay_days",
    "pred_b1r_source",
]
current_open_baseline.loc[:, [c for c in current_open_priority if c in current_open_baseline.columns]].to_csv(
    CURRENT_OPEN_BASELINE_PATH, index=False
)

print(f"Saved baseline snapshot predictions: {BASELINE_PREDICTIONS_PATH}")
print(f"Saved current open baseline forecast: {CURRENT_OPEN_BASELINE_PATH}")
baseline_predictions[[c for c in prediction_priority if c in baseline_predictions.columns]].head()

## 3. Invoice-level baseline metrics

`B1` là mốc chính phải beat. `B1R` là challenger recency, hữu ích để kiểm tra drift.

In [ ]:
def invoice_metrics(frame, pred_col, group_col=None):
    if group_col is None:
        groups = [("overall", frame)]
    else:
        groups = frame.groupby(group_col, dropna=False)

    records = []
    for key, group in groups:
        y_true = pd.to_numeric(group["remaining_days_at_T"], errors="coerce")
        y_pred = pd.to_numeric(group[pred_col], errors="coerce")
        valid = y_true.notna() & y_pred.notna()
        y_true = y_true.loc[valid]
        y_pred = y_pred.loc[valid]
        err = y_pred - y_true
        abs_err = err.abs()

        if group_col is None:
            payload = {"scope": "overall"}
        else:
            payload = {group_col: key}

        records.append(
            {
                **payload,
                "n": int(valid.sum()),
                "mae_days": float(abs_err.mean()),
                "median_ae_days": float(abs_err.median()),
                "rmse_days": float(np.sqrt(np.mean(np.square(err)))),
                "bias_days": float(err.mean()),
                "p50_pinball_loss": float(0.5 * abs_err.mean()),
            }
        )
    return pd.DataFrame(records)


invoice_metric_frames = []
for baseline_name, pred_col in BASELINES.items():
    overall = invoice_metrics(baseline_predictions, pred_col)
    split = invoice_metrics(baseline_predictions, pred_col, "time_split")
    overall["baseline"] = baseline_name
    split["baseline"] = baseline_name
    invoice_metric_frames.extend([overall, split])

baseline_invoice_metrics = pd.concat(invoice_metric_frames, ignore_index=True)
metric_order = ["baseline", "scope", "time_split", "n", "mae_days", "median_ae_days", "rmse_days", "bias_days", "p50_pinball_loss"]
baseline_invoice_metrics = baseline_invoice_metrics[[c for c in metric_order if c in baseline_invoice_metrics.columns]]
baseline_invoice_metrics.to_csv(BASELINE_INVOICE_METRICS_PATH, index=False)

baseline_invoice_metrics

## 4. Segment diagnostics

Chỉ nên tin baseline nếu nó vẫn ổn trên overdue, warm/cold customer và high-value invoices.

In [ ]:
def make_segment_frame(frame):
    customer_history = pd.to_numeric(frame["customer_history_sufficient"], errors="coerce").fillna(0).eq(1)
    amount_percentile = pd.to_numeric(frame["amount_percentile_hist"], errors="coerce").fillna(0)
    segment_frames = []
    segment_specs = {
        "overdue_status": np.where(frame["is_overdue_at_T"].eq(1), "overdue", "not_overdue"),
        "customer_history": np.where(customer_history, "warm_customer", "cold_customer"),
        "amount_band": np.where(amount_percentile.ge(0.90), "top_10pct_amount", "normal_amount"),
    }
    for segment_name, segment_values in segment_specs.items():
        segment_frame = frame.copy()
        segment_frame["segment_name"] = segment_name
        segment_frame["segment_value"] = segment_values
        segment_frames.append(segment_frame)
    return pd.concat(segment_frames, ignore_index=True)


segment_eval_frame = make_segment_frame(baseline_predictions)
segment_metric_frames = []
for baseline_name, pred_col in BASELINES.items():
    metrics = invoice_metrics(segment_eval_frame, pred_col, "time_split")
    # expand time_split metrics into segment metrics
    expanded = []
    for _, row in metrics.iterrows():
        for segment_name in ["overdue_status", "customer_history", "amount_band"]:
            subset = segment_eval_frame[
                (segment_eval_frame["time_split"] == row["time_split"]) &
                (segment_eval_frame["segment_name"] == segment_name)
            ]
            y_true = pd.to_numeric(subset["remaining_days_at_T"], errors="coerce")
            y_pred = pd.to_numeric(subset[pred_col], errors="coerce")
            valid = y_true.notna() & y_pred.notna()
            y_true = y_true.loc[valid]
            y_pred = y_pred.loc[valid]
            err = y_pred - y_true
            abs_err = err.abs()
            expanded.append(
                {
                    "baseline": baseline_name,
                    "time_split": row["time_split"],
                    "segment_name": segment_name,
                    "segment_value": "aggregate",
                    "n": int(valid.sum()),
                    "mae_days": float(abs_err.mean()),
                    "median_ae_days": float(abs_err.median()),
                    "rmse_days": float(np.sqrt(np.mean(np.square(err)))),
                    "bias_days": float(err.mean()),
                }
            )
    segment_metric_frames.append(pd.DataFrame(expanded))

baseline_segment_metrics = pd.concat(segment_metric_frames, ignore_index=True)
baseline_segment_metrics.to_csv(BASELINE_SEGMENT_METRICS_PATH, index=False)
baseline_segment_metrics

## 5. Cash baseline và open forecast

Cash-level baseline là lớp aggregate của invoice-level predictions trên 30/60/90 ngày.

In [ ]:
def build_cash_by_cutoff(frame):
    records = []
    for baseline_name, pred_col in BASELINES.items():
        for horizon in HORIZONS:
            temp = frame.copy()
            actual_cash = np.where(
                pd.to_numeric(temp["remaining_days_at_T"], errors="coerce").le(horizon),
                pd.to_numeric(temp["amount"], errors="coerce"),
                0.0,
            )
            predicted_cash = np.where(
                pd.to_numeric(temp[pred_col], errors="coerce").le(horizon),
                pd.to_numeric(temp["amount"], errors="coerce"),
                0.0,
            )
            grouped = (
                temp.assign(actual_cash=actual_cash, predicted_cash=predicted_cash)
                .groupby(["time_split", "cutoff_date"], dropna=False)
                .agg(
                    actual_cash=("actual_cash", "sum"),
                    predicted_cash=("predicted_cash", "sum"),
                    open_amount=("amount", "sum"),
                )
                .reset_index()
            )
            grouped["baseline"] = baseline_name
            grouped["horizon_days"] = horizon
            grouped["cash_abs_error"] = (grouped["predicted_cash"] - grouped["actual_cash"]).abs()
            grouped["cash_error"] = grouped["predicted_cash"] - grouped["actual_cash"]
            records.append(grouped)
    return pd.concat(records, ignore_index=True)


baseline_cash_by_cutoff = build_cash_by_cutoff(baseline_predictions)
baseline_cash_metrics = (
    baseline_cash_by_cutoff.groupby(["baseline", "time_split", "horizon_days"], dropna=False)
    .agg(
        cutoffs=("cutoff_date", "nunique"),
        actual_cash=("actual_cash", "sum"),
        predicted_cash=("predicted_cash", "sum"),
        cash_abs_error=("cash_abs_error", "sum"),
        cash_error=("cash_error", "sum"),
    )
    .reset_index()
)
baseline_cash_metrics["cash_wmape"] = baseline_cash_metrics["cash_abs_error"] / baseline_cash_metrics["actual_cash"].replace(0, np.nan)
baseline_cash_metrics["cash_bias_pct"] = baseline_cash_metrics["cash_error"] / baseline_cash_metrics["actual_cash"].replace(0, np.nan)

baseline_cash_by_cutoff.to_csv(BASELINE_CASH_BY_CUTOFF_PATH, index=False)
baseline_cash_metrics.to_csv(BASELINE_CASH_METRICS_PATH, index=False)

baseline_cash_metrics

## 6. Baseline gate summary

`B1` là mốc chính để LightGBM phải beat. `B1R` là challenger recency, hữu ích để kiểm tra drift. Due-date clamp không còn nằm trong core gate.

In [ ]:
test_invoice_metrics = baseline_invoice_metrics.query("time_split == 'test'").copy()
summary_rows = []
for baseline_name in BASELINES:
    row = test_invoice_metrics.loc[test_invoice_metrics["baseline"].eq(baseline_name)].iloc[0]
    summary_rows.append(
        {
            "baseline": baseline_name,
            "test_mae_days": float(row["mae_days"]),
            "test_median_ae_days": float(row["median_ae_days"]),
            "test_bias_days": float(row["bias_days"]),
        }
    )

gate_summary = pd.DataFrame(summary_rows)
best_baseline = gate_summary.sort_values("test_mae_days").iloc[0]["baseline"]
b1_mae = gate_summary.loc[gate_summary["baseline"].eq("B1_customer_median_backoff"), "test_mae_days"].iloc[0]
b1r_mae = gate_summary.loc[gate_summary["baseline"].eq("B1R_recency_last5_median_backoff"), "test_mae_days"].iloc[0]
b1r_delta = (b1r_mae - b1_mae) / b1_mae

decision_summary = pd.DataFrame(
    [
        ("best_baseline_on_test", best_baseline),
        ("b1_test_mae_days", b1_mae),
        ("b1r_test_mae_days", b1r_mae),
        ("b1r_vs_b1_mae_delta", b1r_delta),
        ("baseline_predictions_path", str(BASELINE_PREDICTIONS_PATH)),
        ("baseline_invoice_metrics_path", str(BASELINE_INVOICE_METRICS_PATH)),
        ("baseline_segment_metrics_path", str(BASELINE_SEGMENT_METRICS_PATH)),
        ("baseline_cash_metrics_path", str(BASELINE_CASH_METRICS_PATH)),
        ("current_open_baseline_path", str(CURRENT_OPEN_BASELINE_PATH)),
    ],
    columns=["item", "value"],
)

print(gate_summary.to_string(index=False))
print(decision_summary.to_string(index=False))